# Anima · ComfyUI Colab
GPU 런타임에서 위에서부터 실행하세요. `LORA_PATH`에 Google Drive의 LoRA 파일 또는 폴더 경로를 입력합니다.

ComfyUI + Manager 설치 → Anima 모델 다운로드 → LoRA 연결 → Cloudflare 접속 링크를 출력합니다. 생성 설정은 ComfyUI에서 조절하세요.

Google Drive에는 저장하지 않습니다. 이미지는 `/content/ComfyUI/output`에 저장됩니다. 터널 링크는 비밀번호 없는 공개 주소이므로 공유하지 마세요.

In [ ]:
#@title 1. Google Drive 연결 · LoRA 경로
LORA_PATH = "/content/drive/MyDrive/Anima/loras" #@param {type:"string"}

from google.colab import drive
from pathlib import Path
import os, sys, json, subprocess, urllib.request, time, re, shutil, tarfile

drive.mount("/content/drive")
COMFY = Path("/content/ComfyUI")
LORA = Path(LORA_PATH.strip()).resolve()
if not LORA.is_relative_to(Path("/content/drive").resolve()) or not LORA.exists():
    raise ValueError("Google Drive의 실제 LoRA 파일 또는 폴더 경로를 입력하세요.")
if LORA.is_file() and LORA.suffix.lower() != ".safetensors":
    raise ValueError("LoRA는 .safetensors 파일을 지정하세요.")
if LORA.is_dir() and not any(LORA.rglob("*.safetensors")):
    raise ValueError("폴더에 .safetensors 파일이 없습니다.")

def run(*args, **kwargs):
    return subprocess.run([str(a) for a in args], check=True, **kwargs)

print("연결할 LoRA:", LORA)


In [ ]:
#@title 2. ComfyUI + Manager 설치
run("nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader")
if not COMFY.exists():
    run("git", "clone", "--depth", "1", "https://github.com/Comfy-Org/ComfyUI.git", COMFY)

UV = Path("/content/uv")
if not UV.exists():
    archive_path = Path("/content/uv.tar.gz")
    urllib.request.urlretrieve("https://github.com/astral-sh/uv/releases/latest/download/uv-x86_64-unknown-linux-gnu.tar.gz", archive_path)
    with tarfile.open(archive_path) as archive:
        member = next(m for m in archive.getmembers() if m.isfile() and m.name.endswith("/uv"))
        UV.write_bytes(archive.extractfile(member).read())
    UV.chmod(0o755)

VENV = Path("/content/comfyui-venv")
PYTHON = VENV / "bin/python"
if not PYTHON.exists():
    run(UV, "venv", "--python", sys.executable, "--seed", "--system-site-packages", VENV)
run(UV, "pip", "install", "--python", PYTHON,
    "-r", COMFY / "requirements.txt", "-r", COMFY / "manager_requirements.txt")
print("ComfyUI + 공식 Manager 설치 완료")


In [ ]:
#@title 3. Anima 모델 다운로드 · LoRA 연결
model_files = {
    "diffusion_models": "anima-base-v1.0.safetensors",
    "text_encoders": "qwen_3_06b_base.safetensors",
    "vae": "qwen_image_vae.safetensors",
}
for folder, name in model_files.items():
    target = COMFY / "models" / folder / name
    target.parent.mkdir(parents=True, exist_ok=True)
    if not target.exists():
        url = f"https://huggingface.co/circlestone-labs/Anima/resolve/main/split_files/{folder}/{name}"
        partial = target.with_suffix(".part")
        print("다운로드:", name, flush=True)
        run("curl", "--fail", "--location", "--retry", "3", "--output", partial, url)
        partial.replace(target)

link = COMFY / "models/loras" / ("from_drive" if LORA.is_dir() else "drive_" + LORA.name)
link.parent.mkdir(parents=True, exist_ok=True)
if link.is_symlink():
    link.unlink()
elif link.exists():
    raise FileExistsError(f"기존 경로를 덮어쓸 수 없습니다: {link}")
link.symlink_to(LORA, target_is_directory=LORA.is_dir())
print("LoRA 연결 완료:", link.name)
print("ComfyUI의 LoRA 로더에서 선택하세요. Drive 원본은 변경하지 않습니다.")


In [ ]:
#@title 4. ComfyUI 실행 · Cloudflare 접속 링크
from IPython.display import display, HTML

# 이 셀을 다시 실행할 때 이전 실행을 종료합니다.
for name in ("tunnel_process", "comfy_process"):
    process = globals().get(name)
    if process is not None and process.poll() is None:
        process.terminate()
        try:
            process.wait(timeout=15)
        except subprocess.TimeoutExpired:
            process.kill()
            process.wait()

CLOUDFLARED = Path("/content/cloudflared")
if not CLOUDFLARED.exists():
    urllib.request.urlretrieve("https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64", CLOUDFLARED)
    CLOUDFLARED.chmod(0o755)

env = {**os.environ, "VIRTUAL_ENV": str(VENV), "PYTHONUNBUFFERED": "1",
       "PATH": str(VENV / "bin") + os.pathsep + os.environ["PATH"]}
with open("/content/comfyui.log", "w") as log:
    comfy_process = subprocess.Popen(
        [str(PYTHON), "main.py", "--listen", "127.0.0.1", "--port", "8188", "--enable-manager"],
        cwd=COMFY, env=env, stdout=log, stderr=subprocess.STDOUT)

try:
    for _ in range(150):
        if comfy_process.poll() is not None:
            raise RuntimeError(Path("/content/comfyui.log").read_text(errors="replace")[-6000:])
        try:
            with urllib.request.urlopen("http://127.0.0.1:8188/system_stats", timeout=3):
                break
        except (OSError, TimeoutError):
            time.sleep(2)
    else:
        raise TimeoutError("ComfyUI 시작 시간 초과. /content/comfyui.log를 확인하세요.")

    with open("/content/cloudflared.log", "w") as log:
        tunnel_process = subprocess.Popen(
            [str(CLOUDFLARED), "tunnel", "--url", "http://127.0.0.1:8188", "--protocol", "http2", "--no-autoupdate"],
            stdout=log, stderr=subprocess.STDOUT)
    for _ in range(120):
        log_text = Path("/content/cloudflared.log").read_text(errors="replace")
        if tunnel_process.poll() is not None:
            raise RuntimeError(log_text[-6000:])
        match = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", log_text)
        if match and "Registered tunnel connection" in log_text:
            display(HTML(f'<a href="{match.group(0)}" target="_blank" rel="noopener">ComfyUI 열기 ↗</a>'))
            print("워크플로와 LoRA 강도 등은 ComfyUI에서 설정하세요.")
            print("이미지 저장: /content/ComfyUI/output")
            break
        time.sleep(1)
    else:
        raise TimeoutError("터널 연결 시간 초과. /content/cloudflared.log를 확인하세요.")
except BaseException:
    for name in ("tunnel_process", "comfy_process"):
        process = globals().get(name)
        if process is not None and process.poll() is None:
            process.terminate()
    raise
